# 🧠 Poker Arena — Notebook 02: NFSP, a IA que *aprende* (rede neural)

No notebook 01 o **CFR** montava uma *tabela* de estratégia. Aqui é diferente: uma **rede neural** (NFSP — Neural Fictitious Self-Play) **aprende jogando contra si mesma**, igual um cérebro aprende com a experiência. Você vê a *brecha* (exploitability) cair conforme ela aprende.

**Rode a única célula abaixo.** Roda em poucos minutos (no seu **Colab Pro+** dá pra ir muito além — só aumentar `NUM_EPISODES`).

In [ ]:
# ============================================================
#  POKER ARENA - Notebook 02: NFSP (a IA que APRENDE por rede neural)
#  Estrutura fiel ao exemplo oficial nfsp_leduc_pytorch.py do OpenSpiel.
#  (Se falhar no 1o import, rode a celula de novo: instalou agora.)
# ============================================================

!pip install -q open_spiel

import math
from open_spiel.python import policy, rl_environment
from open_spiel.python.algorithms import exploitability
from open_spiel.python.pytorch import nfsp
import matplotlib.pyplot as plt

# ---------- helpers de log amigavel (estilo 'Use a Cabeca') ----------
def box(titulo, linhas):
    largura = max([len(titulo)] + [len(l) for l in linhas])
    print('  +' + '-' * (largura + 2) + '+')
    print('  | ' + titulo.ljust(largura) + ' |')
    print('  +' + '-' * (largura + 2) + '+')
    for l in linhas:
        print('  | ' + l.ljust(largura) + ' |')
    print('  +' + '-' * (largura + 2) + '+')

def comentario(e):
    if e > 1.0:  return 'jogando quase aleatorio - acabou de nascer'
    if e > 0.3:  return 'pegando as regras...'
    if e > 0.05: return 'opa, ta ficando esperto!'
    return 'quase imbativel!'

def barra(e, e0):
    lo, hi = math.log10(0.01), math.log10(max(e0, 0.01) + 1e-12)
    cur = math.log10(max(e, 1e-9))
    frac = max(0.0, min(1.0, (cur - lo) / (hi - lo + 1e-12)))
    n = int(frac * 20)
    return '[' + '#' * n + '.' * (20 - n) + ']'

NUM_PLAYERS = 2
GAME = 'leduc_poker'           # troque por 'kuhn_poker' p/ convergir mais rapido
NUM_EPISODES = 30000           # aumente p/ brecha menor (Colab Pro+ aguenta muito mais)
EVAL_EVERY = 3000

# wrapper que transforma os agentes NFSP numa policy avaliavel (do exemplo oficial)
class NFSPPolicies(policy.Policy):
    def __init__(self, env, nfsp_policies, mode):
        super().__init__(env.game, list(range(NUM_PLAYERS)))
        self._policies = nfsp_policies
        self._mode = mode
        self._obs = {'info_state': [None] * NUM_PLAYERS,
                     'legal_actions': [None] * NUM_PLAYERS}
    def action_probabilities(self, state, player_id=None):
        cur = state.current_player()
        legal = state.legal_actions(cur)
        self._obs['current_player'] = cur
        self._obs['info_state'][cur] = state.information_state_tensor(cur)
        self._obs['legal_actions'][cur] = legal
        ts = rl_environment.TimeStep(observations=self._obs, rewards=None,
                                     discounts=None, step_type=None)
        with self._policies[cur].temp_mode_as(self._mode):
            p = self._policies[cur].step(ts, is_evaluation=True).probs
        return {a: p[a] for a in legal}

# ---------- intro ----------
box('NOTEBOOK 02 - A IA QUE APRENDE (NFSP)', [
    'No notebook 01, o CFR montava uma TABELA gigante de estrategia.',
    'Aqui e diferente: uma REDE NEURAL aprende jogando contra si',
    'mesma - igual um cerebro aprende com a experiencia.',
])
box('NAO TEM PERGUNTA IDIOTA', [
    'P: O que e NFSP?',
    'R: Neural Fictitious Self-Play. A rede joga, erra e ajusta os',
    '   pesos pra errar menos. E o metodo classico de deep RL p/ poker.',
    'P: Por que a brecha cai mais devagar que no CFR?',
    'R: RL e mais manhoso (ruidoso). Aprender por experiencia leva',
    '   mais tempo que calcular a tabela - mas e mais geral.',
])

# ---------- monta ambiente + agentes ----------
env = rl_environment.Environment(GAME)
info_state_size = env.observation_spec()['info_state'][0]
num_actions = env.action_spec()['num_actions']
agents = [
    nfsp.NFSP(idx, info_state_size, num_actions,
              hidden_layers_sizes=[128],
              reservoir_buffer_capacity=int(2e6),
              anticipatory_param=0.1,
              replay_buffer_capacity=int(2e5),
              min_buffer_size_to_learn=1000,
              batch_size=128, learn_every=64,
              rl_learning_rate=0.01, sl_learning_rate=0.01,
              optimizer_str='sgd')
    for idx in range(NUM_PLAYERS)
]
joint_avg_policy = NFSPPolicies(env, agents, nfsp.MODE.AVERAGE_POLICY)

# ---------- treino narrado ----------
box('TREINANDO NFSP em ' + GAME, [
    'A rede vai jogar ' + str(NUM_EPISODES) + ' maos contra si mesma.',
    'Acompanhe a BRECHA encolhendo conforme ela aprende:',
    '(leva alguns minutos; com Pro+ da pra ir muito alem)',
])
its, conv, e0 = [], [], None
for ep in range(1, NUM_EPISODES + 1):
    ts = env.reset()
    while not ts.last():
        pid = ts.observations['current_player']
        out = agents[pid].step(ts)
        ts = env.step([out.action])
    for ag in agents:
        ag.step(ts)
    if ep == 1 or ep % EVAL_EVERY == 0:
        e = exploitability.exploitability(env.game, joint_avg_policy)
        if e0 is None:
            e0 = e
        its.append(ep); conv.append(e)
        print(f'   episodio {ep:>6}  brecha={e:7.4f}  {barra(e, e0)}  {comentario(e)}')

# ---------- grafico ----------
print()
print('  Desenhando a curva de aprendizado...')
plt.figure(figsize=(7, 4))
plt.semilogy(its, conv, marker='o', color='#16a085')
plt.title('NFSP: a IA aprende e a brecha cai')
plt.xlabel('episodios de self-play'); plt.ylabel('brecha (log)')
plt.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.savefig('nfsp_learning.png', dpi=150); plt.show()

# ---------- fechamento ----------
print()
box('O QUE VOCE ACABOU DE VER', [
    f'Brecha inicial: {conv[0]:.3f}   ->   final: {conv[-1]:.3f}',
    'Uma REDE NEURAL aprendeu a jogar SOZINHA, so jogando contra si.',
    'A brecha cai mais devagar que no CFR (RL e ruidoso), mas a',
    'tendencia e clara. Mais episodios = mais perto do otimo.',
    'Proximo: Deep CFR / PPO, e o Modo Laboratorio comparando todos.',
])